In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal
import scipy.stats as stats
from matplotlib.pylab import norm


NB_ID = "13"

In [ ]:
def load_barrage_data():
    
    print(f"Loading {MIT_BARRAGE_X.name}...")
    X = np.load(MIT_BARRAGE_X)
    Y = np.load(MIT_BARRAGE_Y)
    df = pd.read_csv(MIT_BARRAGE_DATASET_METADATA_FILE)
    
    return X, Y, df

X_barr, Y_barr, df_meta = load_barrage_data()

print(f"--- Loaded ---")
print(f"Mixtures: {X_barr.shape}")
print(f"Targets:  {Y_barr.shape}")
print(f"Metadata: {len(df_meta)} rows")

In [ ]:
def get_idx(df, target_sinr):
    return df[df['sinr_db'] == target_sinr].index[0]

idx_heavy = get_idx(df_meta, -12) # Heavy Noise
idx_light = get_idx(df_meta, 3)   # Light Noise

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# --- Row 1: Heavy Jamming (-12dB) ---
ax_t = axes[0, 0]
ax_f = axes[0, 1]

# Time Domain (Zoomed)
ax_t.plot(X_barr[idx_heavy].real[:200], color='C0', label='Mixture (Noise Dominated)')
ax_t.plot(Y_barr[idx_heavy].real[:200], color='C1', alpha=0.7, label='Clean Signal')
ax_t.set_title("Heavy Barrage (-12dB) - Time")
ax_t.legend()

# Freq Domain (PSD)
ax_f.psd(Y_barr[idx_heavy], NFFT=1024, Fs=1.0, color='C1', label='Clean')
ax_f.psd(X_barr[idx_heavy], NFFT=1024, Fs=1.0, color='C0', alpha=0.5, label='Mixture')
ax_f.set_title("Heavy Barrage (-12dB) - Frequency")
ax_f.legend()

# --- Row 2: Light Jamming (+3dB) ---
ax_t = axes[1, 0]
ax_f = axes[1, 1]

ax_t.plot(X_barr[idx_light].real[:200], color='C0', label='Mixture')
ax_t.plot(Y_barr[idx_light].real[:200], color='C1', alpha=0.7, label='Clean Signal')
ax_t.set_title("Light Barrage (+3dB) - Time")
ax_t.legend()

ax_f.psd(Y_barr[idx_light], NFFT=1024, Fs=1.0, color='C1', label='Clean')
ax_f.psd(X_barr[idx_light], NFFT=1024, Fs=1.0, color='C0', alpha=0.5, label='Mixture')
ax_f.set_title("Light Barrage (+3dB) - Frequency")
ax_f.legend()

plt.tight_layout()
save_plot("barrage_jamming_examples", machine_id="B", nb_id=NB_ID, fig_id="01")
plt.show()

In [ ]:
Noise_est = X_barr - Y_barr

# Calculate Power
P_clean = np.mean(np.abs(Y_barr)**2, axis=1)
P_noise = np.mean(np.abs(Noise_est)**2, axis=1)

# Measured SINR
sinr_measured = 10 * np.log10(P_clean / (P_noise + 1e-12))

# Plot
plt.figure(figsize=(6, 6))
sns.scatterplot(x=df_meta['sinr_db'], y=sinr_measured, alpha=0.3)
plt.plot([-15, 5], [-15, 5], 'r--', label='Ideal')
plt.title("Physics Check: Target vs Measured SINR")
plt.xlabel("Target (dB)")
plt.ylabel("Measured (dB)")
plt.grid(True)
save_plot("barrage_sinr_physics_check", machine_id="B", nb_id=NB_ID, fig_id="02")
plt.show()

# Error
error = np.mean(np.abs(df_meta['sinr_db'] - sinr_measured))
print(f"Mean SINR Error: {error:.4f} dB")
assert error < 0.1, "CRITICAL: Barrage mixing math is wrong!"

In [ ]:
def calc_sfm(signals):
    sfm_list = []
    for sig in signals:
        # Welch's PSD
        f, psd = signal.welch(sig, fs=1.0, nperseg=1024)
        psd = psd[1:] # Remove DC
        
        # SFM = Geometric Mean / Arithmetic Mean
        sfm = gmean(psd) / (np.mean(psd) + 1e-12)
        sfm_list.append(sfm)
    return np.array(sfm_list)

print("Calculating Spectral Flatness (Subset)...")
subset_n = 500
sfm_vals = calc_sfm(X_barr[:subset_n])
meta_sub = df_meta.iloc[:subset_n]

plt.figure(figsize=(10, 6))
sns.boxplot(data=meta_sub, x='sinr_db', y=sfm_vals)
plt.title("Barrage Jamming: Spectral Flatness vs SINR")
plt.xlabel("SINR (dB)")
plt.ylabel("Spectral Flatness (1.0 = Flat Noise)")
plt.grid(True, alpha=0.3)

save_plot("barrage_flatness_trend", machine_id="B",nb_id=NB_ID,fig_id="03")
plt.show()